In [1]:
import sqlite3
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor

In [2]:
# 1) Ler dados
# ================================

db_path = r"C:\Users\beasa\Desktop\AASE\ScreenTimevsMentalWellness.db"
table_main = "ScreenTimevsMentalWellness"

conn = sqlite3.connect(db_path)
df = pd.read_sql_query(f"SELECT * FROM {table_main}", conn)
conn.close()

print("Dimensão inicial:", df.shape)

target = "mental_wellness_index_0_100"

if target not in df.columns:
    raise Exception("Coluna alvo não encontrada.")

y = df[target]
mask = y.notna()
df = df[mask]
y = y[mask]


Dimensão inicial: (399, 20)


In [3]:
# 2) Features do Cenário B
#    (sem stress e produtividade)
# ================================

features_B = [
    "screen_time_hours",
    "work_screen_hours",
    "leisure_screen_hours",
    "sleep_hours",
    "sleep_quality_1_5",
    "exercise_minutes_per_week",
    "social_hours_per_week",
    # sem "stress_level_0_10"
    # sem "productivity_0_100"
    "occupation_Desempregado",
    "occupation_Empregado",
    "occupation_Estudante",
    "occupation_Reformado",
    "work_mode_Hybrid",
    "work_mode_In-person",
    "work_mode_Remote",
    "age_Jovem_16-25",
    "age_Adulto_25-50",
    "age_Senior_50-60"
]

features_B = [c for c in features_B if c in df.columns]
X = df[features_B].fillna(df[features_B].mean())

print("Features usadas no Cenário B:", features_B)
print("Dimensão do X:", X.shape)

Features usadas no Cenário B: ['screen_time_hours', 'work_screen_hours', 'leisure_screen_hours', 'sleep_hours', 'sleep_quality_1_5', 'exercise_minutes_per_week', 'social_hours_per_week', 'occupation_Desempregado', 'occupation_Empregado', 'occupation_Estudante', 'occupation_Reformado', 'work_mode_Hybrid', 'work_mode_In-person', 'work_mode_Remote', 'age_Jovem_16-25', 'age_Adulto_25-50', 'age_Senior_50-60']
Dimensão do X: (399, 17)


In [4]:
# 3) Modelos – 5 técnicas
# ================================

scaler = StandardScaler()

models = {
    "LinearRegression": Pipeline([
        ("scaler", scaler),
        ("model", LinearRegression())
    ]),

    "RandomForest": Pipeline([
        ("scaler", scaler),
        ("model", RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        ))
    ]),

    "DecisionTree": Pipeline([
        ("scaler", scaler),
        ("model", DecisionTreeRegressor(random_state=42))
    ]),

    "AdaBoost": Pipeline([
        ("scaler", scaler),
        ("model", AdaBoostRegressor(
            estimator=DecisionTreeRegressor(max_depth=4),
            n_estimators=200,
            random_state=42
        ))
    ]),

    "GradientBoosting": Pipeline([
        ("scaler", scaler),
        ("model", GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.05,
            random_state=42
        ))
    ])
}

In [5]:
# ================================
# 4) Cross-Validation 10-fold
# ================================

kfold = KFold(n_splits=10, shuffle=True, random_state=42)

print("\n===== CENÁRIO B — 5 Técnicas de Regressão (sem stress/produtividade) =====\n")

results = []

for name, pipe in models.items():
    print(f"A avaliar modelo: {name}...")
    
    cv = cross_validate(
        pipe, X, y, cv=kfold,
        scoring={
            "MAE": "neg_mean_absolute_error",
            "RMSE": "neg_root_mean_squared_error",
            "R2": "r2"
        }
    )

    mae = -cv["test_MAE"].mean()
    rmse = -cv["test_RMSE"].mean()
    r2 = cv["test_R2"].mean()

    print(f"Modelo: {name}")
    print(f" MAE:  {mae:.3f}")
    print(f" RMSE: {rmse:.3f}")
    print(f" R2:   {r2:.3f}\n")

    results.append({
        "Cenário": "B",
        "Modelo": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

df_results = pd.DataFrame(results)


===== CENÁRIO B — 5 Técnicas de Regressão (sem stress/produtividade) =====

A avaliar modelo: LinearRegression...
Modelo: LinearRegression
 MAE:  7.789
 RMSE: 9.951
 R2:   0.749

A avaliar modelo: RandomForest...
Modelo: RandomForest
 MAE:  8.063
 RMSE: 10.390
 R2:   0.723

A avaliar modelo: DecisionTree...
Modelo: DecisionTree
 MAE:  10.450
 RMSE: 14.132
 R2:   0.481

A avaliar modelo: AdaBoost...
Modelo: AdaBoost
 MAE:  8.951
 RMSE: 10.975
 R2:   0.694

A avaliar modelo: GradientBoosting...
Modelo: GradientBoosting
 MAE:  8.295
 RMSE: 10.950
 R2:   0.697



In [6]:
# 5) Tabela final bonita
# ================================

df_results["MAE"] = df_results["MAE"].round(3)
df_results["RMSE"] = df_results["RMSE"].round(3)
df_results["R2"] = df_results["R2"].round(3)

df_results = df_results.sort_values(by="RMSE")

df_results = df_results.rename(columns={
    "Cenário": "Cenário",
    "Modelo": "Modelo",
    "MAE": "MAE",
    "RMSE": "RMSE",
    "R2": "R²"
})

print("\nResultados finais Cenário B:\n")
print(df_results.to_string(index=False, justify="center"))


Resultados finais Cenário B:

Cenário      Modelo        MAE   RMSE    R² 
   B    LinearRegression  7.789  9.951 0.749
   B        RandomForest  8.063 10.390 0.723
   B    GradientBoosting  8.295 10.950 0.697
   B            AdaBoost  8.951 10.975 0.694
   B        DecisionTree 10.450 14.132 0.481
